In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Base paths
BASE = Path(r"C:\Users\Hello\OneDrive\Desktop\Bluestock_internship\bluestock_mf_capstone")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
DB = BASE / "data" / "db"

# Create folders if not exist
PROCESSED.mkdir(parents=True, exist_ok=True)
DB.mkdir(parents=True, exist_ok=True)

print("✅ Paths ready")
print(f"RAW folder: {RAW}")

✅ Paths ready
RAW folder: C:\Users\Hello\OneDrive\Desktop\Bluestock_internship\bluestock_mf_capstone\data\raw


Step 2 — Load all 10 CSVs 

In [3]:
# Load all 10 datasets
datasets = {
    "fund_master":          pd.read_csv(RAW / "01_fund_master.csv"),
    "nav_history":          pd.read_csv(RAW / "02_nav_history.csv"),
    "aum_by_fund_house":    pd.read_csv(RAW / "03_aum_by_fund_house.csv"),
    "monthly_sip_inflows":  pd.read_csv(RAW / "04_monthly_sip_inflows.csv"),
    "category_inflows":     pd.read_csv(RAW / "05_category_inflows.csv"),
    "industry_folio_count": pd.read_csv(RAW / "06_industry_folio_count.csv"),
    "scheme_performance":   pd.read_csv(RAW / "07_scheme_performance.csv"),
    "investor_transactions":pd.read_csv(RAW / "08_investor_transactions.csv"),
    "portfolio_holdings":   pd.read_csv(RAW / "09_portfolio_holdings.csv"),
    "benchmark_indices":    pd.read_csv(RAW / "10_benchmark_indices.csv"),
}

# Print shape of each
for name, df in datasets.items():
    print(f"{name:30s} → {df.shape[0]:>6} rows × {df.shape[1]:>2} cols")

fund_master                    →     40 rows × 15 cols
nav_history                    →  46000 rows ×  3 cols
aum_by_fund_house              →     90 rows ×  5 cols
monthly_sip_inflows            →     48 rows ×  6 cols
category_inflows               →    144 rows ×  3 cols
industry_folio_count           →     21 rows ×  6 cols
scheme_performance             →     40 rows × 19 cols
investor_transactions          →  32778 rows × 13 cols
portfolio_holdings             →    322 rows ×  8 cols
benchmark_indices              →   8050 rows ×  3 cols


Step 3 — Clean nav_history

In [4]:
# ---- Clean nav_history ----
nav = datasets["nav_history"].copy()

print("BEFORE cleaning:")
print(nav.dtypes)
print(nav.isnull().sum())
print(nav.head())

BEFORE cleaning:
amfi_code      int64
date             str
nav          float64
dtype: object
amfi_code    0
date         0
nav          0
dtype: int64
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [8]:
# ---- Clean nav_history ----
nav = datasets["nav_history"].copy()

# 1. Convert date to datetime
nav["date"] = pd.to_datetime(nav["date"])

# 2. Sort by fund and date
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# 3. Reindex each fund to full date range and forward-fill
full_dates = pd.date_range(nav["date"].min(), nav["date"].max(), freq="D")

chunks = []
for code, group in nav.groupby("amfi_code"):
    group = group.set_index("date").reindex(full_dates)
    group["amfi_code"] = code          # restore the code
    group = group.ffill()              # fill NAV gaps
    group.index.name = "date"
    group = group.reset_index()
    chunks.append(group)

nav_clean = pd.concat(chunks, ignore_index=True)

# 4. Remove duplicates
nav_clean = nav_clean.drop_duplicates(subset=["amfi_code", "date"])

# 5. Validate
invalid = nav_clean[nav_clean["nav"] <= 0]
print(f"Invalid NAV rows (should be 0): {len(invalid)}")
print(f"\nAFTER cleaning:")
print(f"Rows: {len(nav_clean)}")
print(f"Date range: {nav_clean['date'].min()} → {nav_clean['date'].max()}")
print(f"Null values:\n{nav_clean.isnull().sum()}")
print(nav_clean.head())

Invalid NAV rows (should be 0): 0

AFTER cleaning:
Rows: 64320
Date range: 2022-01-03 00:00:00 → 2026-05-29 00:00:00
Null values:
date         0
amfi_code    0
nav          0
dtype: int64
        date  amfi_code       nav
0 2022-01-03     100016  520.4608
1 2022-01-04     100016  515.0971
2 2022-01-05     100016  521.7239
3 2022-01-06     100016  515.7880
4 2022-01-07     100016  515.1639


Step 4 — Save clean NAV to processed folder

VV Imp code to save the processed data into our processed file in our folder structure by giving the path

In [9]:
# Save clean nav to processed folder
nav_clean.to_csv(PROCESSED / "clean_nav.csv", index=False)
print(f"✅ clean_nav.csv saved → {len(nav_clean)} rows")

✅ clean_nav.csv saved → 64320 rows


Step 5 — Clean investor_transactions

In [13]:
# ---- Clean investor_transactions ----
txn = datasets["investor_transactions"].copy()

print("BEFORE cleaning:")
print(txn.dtypes)
print(f"\nNull values:\n{txn.isnull().sum()}")
print(f"\nUnique transaction types: {txn['transaction_type'].unique()}")
print(f"Any amount <= 0: {(txn['amount_inr'] <= 0).sum()}")
print(f"KYC status values: {txn['kyc_status'].unique()}")
df.head()

BEFORE cleaning:
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object

Null values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Unique transaction types: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
Any amount <= 0: 0
KYC status values: <StringArray>
['Verified', 'Pending']
Length: 2, dtype: str


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


To a computer, a date string is just a sequence of characters. Without parsing, the computer cannot tell if "05/06/2026" means May 6th or June 5th. Parsing forces the system to translate the text using an exact map (e.g., Month/Day/Year) so it creates a functional date timestamp

In [12]:
# ---- Clean investor_transactions ----
txn = datasets["investor_transactions"].copy()

# 1. Convert date to datetime
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])

# 2. Standardise transaction_type (capitalise properly)
txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()
print(f"Transaction types after cleaning: {txn['transaction_type'].unique()}")

# 3. Validate amount > 0 (already confirmed but let's be safe)
txn = txn[txn["amount_inr"] > 0]

# 4. Remove duplicates
before = len(txn)
txn = txn.drop_duplicates()
print(f"Duplicates removed: {before - len(txn)}")

# 5. Save
txn.to_csv(PROCESSED / "clean_transactions.csv", index=False)
print(f"\n✅ clean_transactions.csv saved → {len(txn)} rows")
df.head()

Transaction types after cleaning: <StringArray>
['Sip', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
Duplicates removed: 0

✅ clean_transactions.csv saved → 32778 rows


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [14]:
# Fix SIP capitalisation
txn["transaction_type"] = txn["transaction_type"].replace("Sip", "SIP")
print(f"Transaction types fixed: {txn['transaction_type'].unique()}")

# Re-save
txn.to_csv(PROCESSED / "clean_transactions.csv", index=False)
print(f"✅ clean_transactions.csv re-saved → {len(txn)} rows")

Transaction types fixed: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
✅ clean_transactions.csv re-saved → 32778 rows


Step 6 — Clean scheme_performance

In [15]:
# ---- Clean scheme_performance ----
perf = datasets["scheme_performance"].copy()

print("BEFORE cleaning:")
print(perf.dtypes)
print(f"\nNull values:\n{perf.isnull().sum()}")

BEFORE cleaning:
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

Null values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
e

Zero nulls, all correct dtypes! Very clean dataset. Let's just validate the numeric ranges and save.


In [16]:
# ---- Clean scheme_performance ----
perf = datasets["scheme_performance"].copy()

# 1. Flag negative Sharpe ratios (just flag, not remove)
negative_sharpe = perf[perf["sharpe_ratio"] < 0]
print(f"Funds with negative Sharpe: {len(negative_sharpe)}")
if len(negative_sharpe) > 0:
    print(negative_sharpe[["scheme_name", "sharpe_ratio"]])

# 2. Validate expense ratio range (should be 0.1% to 2.5%)
invalid_expense = perf[(perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)]
print(f"\nFunds with unusual expense ratio: {len(invalid_expense)}")

# 3. Validate return values are numeric (already confirmed but document it)
print(f"\nReturn columns are numeric: {perf[['return_1yr_pct','return_3yr_pct','return_5yr_pct']].dtypes.unique()}")

# 4. Remove duplicates
perf = perf.drop_duplicates()

# 5. Save
perf.to_csv(PROCESSED / "clean_performance.csv", index=False)
print(f"\n✅ clean_performance.csv saved → {len(perf)} rows")

Funds with negative Sharpe: 0

Funds with unusual expense ratio: 0

Return columns are numeric: [dtype('float64')]

✅ clean_performance.csv saved → 40 rows


Perfect! ✅ All 40 funds clean, no issues at all.

Step 7 — Cleaning remaining 7 datasets in one go

In [17]:
# ---- Clean remaining 7 datasets ----

# 1. fund_master
fm = datasets["fund_master"].copy()
fm["launch_date"] = pd.to_datetime(fm["launch_date"])
fm = fm.drop_duplicates()
fm.to_csv(PROCESSED / "clean_fund_master.csv", index=False)
print(f"✅ clean_fund_master.csv → {len(fm)} rows")

# 2. aum_by_fund_house
aum = datasets["aum_by_fund_house"].copy()
aum = aum.drop_duplicates()
aum.to_csv(PROCESSED / "clean_aum_by_fund_house.csv", index=False)
print(f"✅ clean_aum_by_fund_house.csv → {len(aum)} rows")

# 3. monthly_sip_inflows
sip = datasets["monthly_sip_inflows"].copy()
sip = sip.drop_duplicates()
sip.to_csv(PROCESSED / "clean_monthly_sip_inflows.csv", index=False)
print(f"✅ clean_monthly_sip_inflows.csv → {len(sip)} rows")

# 4. category_inflows
cat = datasets["category_inflows"].copy()
cat = cat.drop_duplicates()
cat.to_csv(PROCESSED / "clean_category_inflows.csv", index=False)
print(f"✅ clean_category_inflows.csv → {len(cat)} rows")

# 5. industry_folio_count
folio = datasets["industry_folio_count"].copy()
folio = folio.drop_duplicates()
folio.to_csv(PROCESSED / "clean_industry_folio_count.csv", index=False)
print(f"✅ clean_industry_folio_count.csv → {len(folio)} rows")

# 6. portfolio_holdings
port = datasets["portfolio_holdings"].copy()
port = port.drop_duplicates()
port.to_csv(PROCESSED / "clean_portfolio_holdings.csv", index=False)
print(f"✅ clean_portfolio_holdings.csv → {len(port)} rows")

# 7. benchmark_indices
bench = datasets["benchmark_indices"].copy()
bench["date"] = pd.to_datetime(bench["date"])
bench = bench.sort_values("date").drop_duplicates()
bench.to_csv(PROCESSED / "clean_benchmark_indices.csv", index=False)
print(f"✅ clean_benchmark_indices.csv → {len(bench)} rows")

✅ clean_fund_master.csv → 40 rows
✅ clean_aum_by_fund_house.csv → 90 rows
✅ clean_monthly_sip_inflows.csv → 48 rows
✅ clean_category_inflows.csv → 144 rows
✅ clean_industry_folio_count.csv → 21 rows
✅ clean_portfolio_holdings.csv → 322 rows
✅ clean_benchmark_indices.csv → 8050 rows


Step 8 — Create the database and load all tables

In [18]:
from sqlalchemy import create_engine

# Create SQLite database
db_path = DB / "bluestock_mf.db"
engine = create_engine(f"sqlite:///{db_path}")

# Load all cleaned datasets into SQLite
nav_clean.to_sql("fact_nav", engine, if_exists="replace", index=False)
print(f"✅ fact_nav loaded → {len(nav_clean)} rows")

txn.to_sql("fact_transactions", engine, if_exists="replace", index=False)
print(f"✅ fact_transactions loaded → {len(txn)} rows")

perf.to_sql("fact_performance", engine, if_exists="replace", index=False)
print(f"✅ fact_performance loaded → {len(perf)} rows")

fm.to_sql("dim_fund", engine, if_exists="replace", index=False)
print(f"✅ dim_fund loaded → {len(fm)} rows")

aum.to_sql("fact_aum", engine, if_exists="replace", index=False)
print(f"✅ fact_aum loaded → {len(aum)} rows")

sip.to_sql("fact_sip_industry", engine, if_exists="replace", index=False)
print(f"✅ fact_sip_industry loaded → {len(sip)} rows")

cat.to_sql("fact_category_inflows", engine, if_exists="replace", index=False)
print(f"✅ fact_category_inflows loaded → {len(cat)} rows")

folio.to_sql("fact_folio_count", engine, if_exists="replace", index=False)
print(f"✅ fact_folio_count loaded → {len(folio)} rows")

port.to_sql("fact_portfolio", engine, if_exists="replace", index=False)
print(f"✅ fact_portfolio loaded → {len(port)} rows")

bench.to_sql("fact_benchmark", engine, if_exists="replace", index=False)
print(f"✅ fact_benchmark loaded → {len(bench)} rows")

print("\n🎉 bluestock_mf.db created successfully!")

✅ fact_nav loaded → 64320 rows
✅ fact_transactions loaded → 32778 rows
✅ fact_performance loaded → 40 rows
✅ dim_fund loaded → 40 rows
✅ fact_aum loaded → 90 rows
✅ fact_sip_industry loaded → 48 rows
✅ fact_category_inflows loaded → 144 rows
✅ fact_folio_count loaded → 21 rows
✅ fact_portfolio loaded → 322 rows
✅ fact_benchmark loaded → 8050 rows

🎉 bluestock_mf.db created successfully!


Step 9 — Writing the schema.sql file

This is required for GitHub (since we don't push the .db file). In my sql/ folder, create a new file called schema.sql

verify the queries actually run against our database.

In [21]:
import sqlite3

# Connect to database
conn = sqlite3.connect(db_path)

# Test Q1: Top 5 funds by AUM
print("Q1: Top 5 funds by AUM")
print("-" * 50)
q1 = pd.read_sql("""
    SELECT d.scheme_name, d.fund_house, p.aum_crore
    FROM fact_performance p
    JOIN dim_fund d USING (amfi_code)
    ORDER BY p.aum_crore DESC
    LIMIT 5
""", conn)
print(q1.to_string())

Q1: Top 5 funds by AUM
--------------------------------------------------
                                             scheme_name         fund_house  aum_crore
0  Mirae Asset Emerging Bluechip Fund - Regular - Growth     Mirae Asset MF      49046
1          Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF      47469
2         Nippon India Small Cap Fund - Regular - Growth    Nippon India MF      43630
3             DSP Top 100 Equity Fund - Regular - Growth    DSP Mutual Fund      41828
4                    UTI Mid Cap Fund - Regular - Growth    UTI Mutual Fund      41728


all 10 queries at once

In [22]:
# all 10 queries and verify
queries = {
    "Q2: Avg NAV per month": """
        SELECT amfi_code, strftime('%Y-%m', date) AS month,
               ROUND(AVG(nav), 4) AS avg_nav
        FROM fact_nav
        GROUP BY amfi_code, month
        ORDER BY amfi_code, month
        LIMIT 5
    """,
    "Q3: SIP inflow YoY growth": """
        SELECT month, sip_inflow_crore, yoy_growth_pct
        FROM fact_sip_industry
        ORDER BY month
        LIMIT 5
    """,
    "Q4: Total transactions by state": """
        SELECT state,
               ROUND(SUM(amount_inr) / 1e7, 2) AS total_amount_crore
        FROM fact_transactions
        GROUP BY state
        ORDER BY total_amount_crore DESC
    """,
    "Q5: Funds with expense ratio < 1%": """
        SELECT scheme_name, fund_house, expense_ratio_pct
        FROM dim_fund
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
        LIMIT 5
    """,
    "Q6: Best funds by 3yr CAGR": """
        SELECT d.scheme_name, p.return_3yr_pct
        FROM fact_performance p
        JOIN dim_fund d USING (amfi_code)
        ORDER BY p.return_3yr_pct DESC
        LIMIT 5
    """,
    "Q7: Avg SIP amount by age group": """
        SELECT age_group, ROUND(AVG(amount_inr), 2) AS avg_sip_amount
        FROM fact_transactions
        WHERE transaction_type = 'SIP'
        GROUP BY age_group
        ORDER BY avg_sip_amount DESC
    """,
    "Q8: Top 5 sectors by portfolio weight": """
        SELECT sector, ROUND(SUM(weight_pct), 2) AS total_weight
        FROM fact_portfolio
        GROUP BY sector
        ORDER BY total_weight DESC
        LIMIT 5
    """,
    "Q9: Funds with Sharpe > 1": """
        SELECT d.scheme_name, p.sharpe_ratio
        FROM fact_performance p
        JOIN dim_fund d USING (amfi_code)
        WHERE p.sharpe_ratio > 1
        ORDER BY p.sharpe_ratio DESC
        LIMIT 5
    """,
    "Q10: SIP vs Lumpsum vs Redemption": """
        SELECT transaction_type,
               COUNT(*) AS total_transactions,
               ROUND(SUM(amount_inr)/1e7, 2) AS total_amount_crore
        FROM fact_transactions
        GROUP BY transaction_type
    """
}

for title, query in queries.items():
    print(f"\n{title}")
    print("-" * 50)
    result = pd.read_sql(query, conn)
    print(result.to_string())

conn.close()
print("\n✅ All 10 queries ran successfully!")


Q2: Avg NAV per month
--------------------------------------------------
   amfi_code    month   avg_nav
0     100016  2022-01  511.9230
1     100016  2022-02  514.5381
2     100016  2022-03  522.2865
3     100016  2022-04  526.1147
4     100016  2022-05  504.3357

Q3: SIP inflow YoY growth
--------------------------------------------------
     month  sip_inflow_crore yoy_growth_pct
0  2022-01             11517           None
1  2022-02             11438           None
2  2022-03             12328           None
3  2022-04             11863           None
4  2022-05             12286           None

Q4: Total transactions by state
--------------------------------------------------
             state  total_amount_crore
0           Punjab               31.58
1       Tamil Nadu               31.52
2   Madhya Pradesh               30.83
3        Rajasthan               29.86
4          Gujarat               29.84
5      West Bengal               29.72
6        Telangana               29